### Artificial Intelligence 464.601 Project #7

### Before You Begin...
00. We're using a Jupyter Notebook environment (tutorial available here: https://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html),
01. Read the entire notebook before beginning your work, and
02.  Check the submission deadline on Gradescope.


### General Directions for this Assignment
00. Output format should be exactly as requested (it is your responsibility to make sure notebook looks as expected on Gradescope), and
01. Functions should do only one thing.


### Before You Submit...
00. Re-read the general instructions provided above, and
01. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
02.  Do not submit any other files.

### Language Modeling

This project will require you to load and train models.  If you choose small models and datasets, you should be able to run this locally on your computer. However, larger models/datasets may require GPU access. You can access one GPU for free on [Google Colab](https://colab.research.google.com/).

We will use HuggingFace libraries. We discussed majority of what you will need during the discussion demo. Additional documentation can be found [here](https://huggingface.co/docs).

In [168]:
# Imports
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import pipeline
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForCausalLM
import os
import pandas as pd
from sklearn.metrics import confusion_matrix
os.environ["WANDB_DISABLED"] = "true"

### Problem 0: Data
From the [HuggingFace Datasets](https://huggingface.co/datasets), choose a dataset that satisfies the following criteria:
- Data must have train and test splits (Optional development set)
- Task must be text classification
- Task must have at least 3 labels


In [169]:
# Loads dataset based on code on dataset page
ds = load_dataset("dair-ai/emotion", "split")

**Describe the data.**
What is the utility of the task? What are the inputs? What are the labels? Are the any potential difficulties you expect from the task? How do you evaluate the performance of this task?

The inputs are a text statement in the form of a first-person statement. The labels are an integer, which corresponds to an emotion, such as 0 meaning "sadness" or 3 meaning "anger." The utility of this text is to train a model that can recognize human emotions.

The utility of this task is that a language model may be able to recognize human emotion, which is a very important aspect to language and communication. Tone and emotion words play a very large part in communication, and this can help a machine to accurately recognize and understand human speech. Also, in the face of people increasingly relying on chatbots for emotional support, it could be incredibly useful for an AI to understand when a speaker is in distress and point them towards apt resources.

I don't necessarily anticipate difficulties with this task itself. I think it will be difficult to train, since the sentences are quite longform and are different than the feature representations I've worked with in the mushroom dataset. However, I don't think that will be difficult to overcome, it will just take work and testing.

I will evaluate performance using the test dataset, to see if the model is able to anticipate emotion on data it has not seen. There are 6 classes, meaning random guesses would yield roughly 17% accuracy. The label split is as follows:

| Emotion | Int | Split |
| ------- | --- | ----- |
| 'joy' | 1 | 33.5% |
| 'sadness' | 0 | 29.5% |
| 'anger' | 3 | 13.5% |
| 'fear' | 4 | 12.1% |
| 'love' | 2 | 8.2% |
| 'surprise' | 5 | 3.6% |

An untrained model would most likely simply guess the highest-prevalence label, being 3: 'joy' at 33.5%. Therefore, I will evaluate the model to be performing a meaningful classification task if it is able to yield a response signfificantly above 33.5%. As a basic benchmark, I will shoot for 50% accuracy or above.

**Research current methods using this dataset.**
What is the current state of the art method? Describe the method, including the type of model used, training protocol (if any), and the performance. Cite your sources.

Lots of the models I've found on HF that use this dataset have been using BERT-like models. These models were quite vague in their descriptions, and the training protocol was a broken link. However, the results of the most popular model I saw using BERT was very promising, with a 0.938 test accuracy using BERT-like language models. This model was the most popular model trained on this dataset: https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion. There were many other models that used BERT and the HF Trainer to yield similar results on the dataset. Here are two more models on HF I looked at to give me some ideas for my own model: https://huggingface.co/nateraw/bert-base-uncased-emotion and https://huggingface.co/bhadresh-savani/albert-base-v2-emotion

Another model I found used transfer learning and looked quite promising. It uses a text-to-text transformer to classify text inputs on the emotion labels from the dataset. The transfer learning involved pre-training on a data-rich task, before fine-tuning the model on downstream tasks. This model produced an average f1-score of 0.83 across the dataset on the test data. https://huggingface.co/mrm8488/t5-base-finetuned-emotion

(Optional) If necessary, perform any data preprocessing here. For example, depending on the dataset you choose, you may need to clean the text or split the training set into a train and validation set.

In [170]:
# Preprocessing and cleaning already completed

### Problem 1: Fine-Tuning a Model
Choose an encoder-only model. Load the model and add a classification layer.

Describe the model you choose. What are the unique properties of this model? What are the pros and cons? Cite your sources.

Based on the results of the BERT-like models I saw, I've selected DistilBERT, which is the model used in the first link. It takes the original model and reduced its size by 40%, while retaining 97% of its language understanding. It uses knowledge distillation to do so, which allows it to acheive similar results. Since this is a BERT-like model, it uses bidirectional attention and encoder-only transformers to process the text bidirectioanlly. It has achieved strong results in sentiment analysis, which corresponds with my task. Since DistilBERT is fine-tuned on the emotion classification dataset, the base encoder keeps pretrained linguistic knowledge, while a newly initialized classification head is trained on the downstream task.

Being on my laptop without access to many GPUs, it achieves high results (the 93.8% accuracy in the first example I found) while using not as much compute power. BERT itself was able to achieve an ever-so-slightly higher result of 94%, but the needed compute power outweighs the marginally improved result, so I selected DistilBERT.

I used the following to get a better understanding of BERT and DistilBERT:
- https://huggingface.co/docs/transformers/model_doc/bert
- https://huggingface.co/bhadresh-savani/distilbert-base-uncased-emotion

Finetune the model on your dataset. Report the performance on the test set.

In [171]:
device = torch.device("cpu")

In [172]:
# Loads DistilBERT tokenizer and applies it to emotions dataset
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding=True)

tokenized_ds = ds.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds.set_format("torch")

In [173]:
# Loads model and automatically adds a classification layer
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6
)

model.to(device)

# NOTE: Status of classifiers being labelled as missing is because I am in the process of training this model on the downstream task of emotional classification, as per this dataset

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12392.32it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [174]:
# Loads data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [175]:
# Creates training and evaluation objects
train_loader = DataLoader(
    tokenized_ds["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    tokenized_ds["test"],
    batch_size=16,
    collate_fn=data_collator
)

In [176]:
def compute_accuracy(model, loader):
    """
    Computes classification accuracy on a dataset.
    Args:
        - model: Trained neural network.
        - data_loader: DataLoader containing batches of data.
    Returns:
        - float: Classification accuracy as a percentage.
    """
    # Sets model to evaluate
    model.eval()
    correct = 0
    total = 0

    # Disables gradient calculations during evaluation
    with torch.no_grad():

        # Processes text in batches
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}

            # Generates predictions
            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)

            # Updates counts
            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    return 100 * correct / total

In [ ]:
epochs = 2
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(epochs):

    # Sets model in training mode
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    # Iterates through batches
    for batch in loop:

        batch = {k: v.to(device) for k, v in batch.items()}

        # Clears gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Computes accuracy
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

    # Computes accuracies
    train_acc = 100 * correct / total
    val_acc = compute_accuracy(model, val_loader)

    # Prints results
    print(f"Loss: {running_loss / len(train_loader):.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%\n")

Epoch 1:  21%|██        | 208/1000 [09:24<36:39,  2.78s/it]

In [127]:
# Sets the model to evaluation mode
model.eval()

all_preds = []
all_labels = []

progress_bar = tqdm(val_loader, desc="Evaluating")

# Disables gradient calculations
with torch.no_grad():
    
    # Resets loss
    running_loss = 0

    # Iterates through batches
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        # Generates predictions
        outputs = model(**batch)
        logits = outputs.logits

        # Computes loss
        loss = outputs.loss
        running_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

Evaluating:  12%|█▏        | 15/125 [00:07<00:52,  2.10it/s]


KeyboardInterrupt: 

In [ ]:
# Compiles and prints results
avg_loss = running_loss / len(val_loader)
accuracy = (np.array(all_preds) == np.array(all_labels)).mean()

print(f"Loss: {avg_loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")

### Problem 2: Error Analysis

Conduct an error analysis on your models. What are your models good at? What do they get wrong? Provide examples of both correct and incorrect predictions. Suggest methods to improve the performance.

In [29]:
# Generates predictions based on evaluation
preds = np.array(all_preds)
labels = np.array(all_labels)

/home/jason-lafita/miniconda3/envs/ai_hw7/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [30]:
# Computes correct vs. incorrect predictions
correct_indices = np.where(preds == labels)[0]
incorrect_indices = np.where(preds != labels)[0]

print("Total correct:", len(correct_indices))
print("Total incorrect:", len(incorrect_indices))

Total correct: 1854
Total incorrect: 146


In [31]:
# Maps label names to feature names
label_names = ds["train"].features["label"].names

In [32]:
# Pulls examples of predictions
def show_examples(indices, num_examples=5):
    for i in indices[:num_examples]:
        text = ds["test"][i]["text"]
        true_label = label_names[labels[i]]
        pred_label = label_names[preds[i]]
        
        print("TEXT:", text)
        print("TRUE:", true_label)
        print("PRED:", pred_label)
        print("-" * 50)

In [33]:
# Displays correct vs. incorrect examples
print("---CORRECT EXAMPLES---")
show_examples(correct_indices)
print("---INCORRECT EXAMPLES---")
show_examples(incorrect_indices)

---CORRECT EXAMPLES---
TEXT: im feeling rather rotten so im not very ambitious right now
TRUE: sadness
PRED: sadness
--------------------------------------------------
TEXT: im updating my blog because i feel shitty
TRUE: sadness
PRED: sadness
--------------------------------------------------
TEXT: i never make her separate from me because i don t ever want her to feel like i m ashamed with her
TRUE: sadness
PRED: sadness
--------------------------------------------------
TEXT: i left with my bouquet of red and yellow tulips under my arm feeling slightly more optimistic than when i arrived
TRUE: joy
PRED: joy
--------------------------------------------------
TEXT: i was feeling a little vain when i did this one
TRUE: sadness
PRED: sadness
--------------------------------------------------
---INCORRECT EXAMPLES---
TEXT: i don t feel particularly agitated
TRUE: fear
PRED: anger
--------------------------------------------------
TEXT: i am right handed however i play billiards left hand

In [44]:
# Visualizes confusion matrix
cm = confusion_matrix(labels, preds)
cm_df = pd.DataFrame(cm, index=label_names, columns=label_names)
cm_df

,sadness,joy,love,anger,fear,surprise
sadness,564,3,1,9,4,0
joy,4,653,34,2,0,2
love,0,16,142,1,0,0
anger,9,3,0,256,7,0
fear,6,0,0,8,197,13
surprise,1,7,0,2,14,42


In [45]:
# Pulls most confused pairs
for i in range(len(label_names)):
    for j in range(len(label_names)):
        if i != j and cm[i][j] > 0:
            print(f"{label_names[i]} -> {label_names[j]}: {cm[i][j]}")

sadness -> joy: 3
sadness -> love: 1
sadness -> anger: 9
sadness -> fear: 4
joy -> sadness: 4
joy -> love: 34
joy -> anger: 2
joy -> surprise: 2
love -> joy: 16
love -> anger: 1
anger -> sadness: 9
anger -> joy: 3
anger -> fear: 7
fear -> sadness: 6
fear -> anger: 8
fear -> surprise: 13
surprise -> sadness: 1
surprise -> joy: 7
surprise -> anger: 2
surprise -> fear: 14


Overall, using DistilBERT on the emotions label dataset produced very consistent, accurate results. The overall accuracy was 92.7%, which is far higher than I initially anticipated I could reach. These results are very similar to the accuracy on the training data, showing that the model does not seem to be falling into the trap of overfitting. This does fall short of the accuracy on the HF model that I based my hyperparameters on, though that experiment ran with 16 epochs, whereas mine only had 2 due to compute limitations. Performing this task with more GPUs using the full BERT model would likely produce better results, likely consistent with the 94% found on the HF model.

Speaking to specific errors, the model was largely accurate, especially in cases of clear emotions being expressed, such as "im feeling rather rotten so im not very ambitious right now" corresponding to sadness. Confusion occurred in emotions with a similar positivity level and intensity level. The largest mixup was joy being confused for love, and vice versa, which makes sense, since both are very positive emotions, and there is frequent overlap in the two in everyday speech. For example, the input "i feel like i am in paradise kissing those sweet lips make me feel like i dive into a magical world of love" was labeled as "joy," despite also being a clear expression of love. The same applies to cases of surprise vs. fear, as both have the same level of intensity towards the circumstance, though surprise has a more positive connotation than fear. This presents an error in the training data, and a systemic issue with classification tasks on the whole, where human speech is difficult to push into little boxes, since lots of natural language has multiple meanings, but a classification task forces a model to choose one label.

Another issue in the dataset that may improve the data is to have a more balanced dataset, where each emotion has equal representation. I could have parsed down many of the emotions to ensure roughly equal data, since that would provide the model with equal understanding of all emotions. There were 17 instances where love was mistaken for something else, which makes up over 10% of the total errors, despite love only being 3.5% of the data. These confusions may be lessened if the data was more balanced.

## Before You Submit...

00. Re-read the general instructions provided above, and
01. Hit "Kernel"->"Restart & Run All". The first cell that is run should show [1], the second should show [2], and so on...
02. Submit your notebook (as .ipynb, not PDF) using Gradescope, and
03.  Do not submit any other files.